# 01 — Harvest all lineage sources

Scheduled by the Fabric Data Pipeline `lineage_harvest_pipeline.json` (default cron: hourly).
Runs every harvester in [`harvesters/`](../harvesters/) and appends the
resulting edges to the `lineage_edges` Delta table in the bound Lakehouse.

Environment (set via Pipeline parameters or workspace KV):

- `TENANT_ID`, `FABRIC_WORKSPACE_ID`, `LAKEHOUSE_ID`
- `ADF_SUBSCRIPTION_ID`, `ADF_RESOURCE_GROUP`, `ADF_FACTORY_NAME`
- `PBI_WORKSPACE_IDS` (comma-separated)
- `TSQL_REPO_ROOT`, `NOTEBOOK_REPO_ROOT` (mounted via git integration)

In [ ]:
%pip install --quiet sqlglot networkx pyvis deltalake

In [ ]:
import os, sys, pathlib
# Add the workspace repo root to sys.path so harvesters/* and common/* resolve.
REPO_ROOT = pathlib.Path("/lakehouse/default/Files/repo/fabric-lineage-graph")
if REPO_ROOT.exists():
    sys.path.insert(0, str(REPO_ROOT))

from harvesters import HARVESTERS
from common.onelake_io import write_edges

In [ ]:
edges = []
for cls in HARVESTERS:
    try:
        # Each harvester reads its own env vars; instantiate with defaults
        h = cls() if cls.__name__ != "TsqlHarvester" else cls(os.environ.get("TSQL_REPO_ROOT", "/lakehouse/default/Files/repo/tsql"))
        produced = list(h.harvest())
        print(f"{cls.name:18} produced {len(produced):>5} edges")
        edges.extend(produced)
    except Exception as exc:
        print(f"{cls.name:18} FAILED — {exc!r}")

print(f"\nTotal edges to write: {len(edges)}")

In [ ]:
n = write_edges(edges)
print(f"Appended {n} rows to lineage_edges Delta table")